# Assignment 5: Neural Networks

---

## Task 2) RNN for Classification

The theses dataset also contains types (diploma, bachelor, master) and categories (internal/external) for each thesis. 
In this part, we want to classify whether the thesis is bachelor or master; and if it's internal or external. 
Since PyTorch provides most things sort-of out of the box, we want you to compare the following Recurrent Neural Network variation: 
[RNN](https://pytorch.org/docs/stable/generated/torch.nn.RNN.html), [GRU](https://pytorch.org/docs/stable/generated/torch.nn.GRU.html), [LSTM](https://pytorch.org/docs/stable/generated/torch.nn.LSTM.html), and Bidirectional-[LSTM](https://pytorch.org/docs/stable/generated/torch.nn.LSTM.html) by using the `bidirectional` flag.
The basic setup as well as some code and steps can be reused from your solution for the language modeling task.

### Data

Download the `theses.csv` data set from the `Supplemental Materials` in the `Files` section of our Microsoft Teams group.
This dataset consists of approx. 3,000 theses topics chosen by students in the past.
Here are some examples of the file content:

```
27.10.94;14.07.95;1995;intern;Diplom;DE;Monte Carlo-Simulation für ein gekoppeltes Round-Robin-System;
04.11.94;14.03.95;1995;intern;Diplom;DE;Implementierung eines Testüberdeckungsgrad-Analysators für RAS;
01.11.20;01.04.21;2021;intern;Bachelor;DE;Landessprachenerkennung mittels X-Vektoren und Meta-Klassifikation;
```

### Basic Setup

For the assignment on Recurrent Neural Networks, we'll (again) heavily use [PyTorch](https://pytorch.org) as go-to Deep Learning library.
Here, we'll rely on the RNN and Embedding modules already implemented by PyTorch.
You can imagine the Embedding layer as a simple lookup table that stores embeddings of a fixed dictionary and size (quite similar to the Word2Vec parameters we've trained in assignment 2).
Head over to the [RNN](https://pytorch.org/docs/stable/generated/torch.nn.RNN.html) and [Embedding](https://pytorch.org/docs/stable/generated/torch.nn.Embedding.html) modules to gain some understanding of their functionality.
Code for processing data samples, batching, converting to tensors, etc. can get messy and hard to maintain. 
Therefore, you can use PyTorch's [Datasets & DataLoaders](https://pytorch.org/tutorials/beginner/basics/data_tutorial.html). 
Get familiar with the basics of data handling, as it will help you for upcoming assignments.
As always, you can use [NumPy](https://numpy.org) and [Pandas](https://pandas.pydata.org) for data handling etc.

*In this Jupyter Notebook, we will provide the steps to solve this task and give hints via functions & comments. However, code modifications (e.g., function naming, arguments) and implementation of additional helper functions & classes are allowed. The code aims to help you get started.*

---

In [1]:
# Dependencies
import os
import re
import tqdm
import string
import numpy as np
import pandas as pd
import sklearn.metrics as sklearn_metrics
from sklearn.model_selection import StratifiedKFold

import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

### Prepare the Data

1.1 Spend some time on preparing the dataset. It may be helpful to lower-case the data and to filter for German titles. The format of the CSV-file should be:

```
Anmeldedatum;Abgabedatum;JahrAkademisch;Art;Grad;Sprache;Titel;Abstract
```

1.2 Create the vocabulary from the prepared dataset. You'll need it for the modeling part such as nn.Embedding.

1.3 Filter out all diploma theses; they might be too easy to spot because they only cover "old" topics.

1.4 Create a PyTorch Dataset class which handles your tokenized data with respect to input and (class) labels.

In [2]:
def load_theses_dataset(filepath):
    """Loads all theses instances and returns them as a dataframe."""
    ### YOUR CODE HERE
    
    return pd.read_csv(filepath, header=0, sep=";")
    
    ### END YOUR CODE

In [3]:
def preprocess(dataframe):
    """Preprocesses and tokenizes the given theses titles for further use."""
    ### YOUR CODE HERE
    
    def _preprocss_fn(text):
        remove_digits = str.maketrans(string.digits, ' '*len(string.digits))
        remove_pun = str.maketrans(string.punctuation, ' '*len(string.punctuation))
        text = text.translate(remove_digits)
        text = text.translate(remove_pun)
        text = re.sub(' {2,}', ' ', text)
        return text.lower()
    
    dataframe = dataframe.copy()
    
    # Remove punctuation, digits and lowercase titles
    dataframe["Titel"] = dataframe["Titel"].apply(lambda s: _preprocss_fn(s))

    # Filter out empty and short titles
    dataframe = dataframe[dataframe["Titel"].str.len() > 4]

    # Reset index of dataframe
    dataframe = dataframe.reset_index(drop=True)

    # Simple tokenization of titles
    dataframe["tokenized"] = [title.split() for title in dataframe["Titel"].values]

    return dataframe

    ### END YOUR CODE

In [4]:
# Load and preprocess dataset
dataframe_all = load_theses_dataset("data/theses2022.csv")
dataframe_all = dataframe_all[dataframe_all["Sprache"] == "DE"]
dataframe_all = preprocess(dataframe_all)

# Convert labels to integer
LABEL2IDX = {"Bachelor": 0, "Master": 1, "Diplom": 2}
dataframe_all["label"] = dataframe_all["Grad"].apply(lambda l: LABEL2IDX[l])

# Filter out `Diplom`
dataframe_diplom = dataframe_all[dataframe_all["Grad"] == "Diplom"]
dataframe = dataframe_all[dataframe_all["Grad"] != "Diplom"]

# Check number of samples and label distribution
print(f"Num theses (overall): {len(dataframe_all)}")
print(f"Num theses (w/o diplom): {len(dataframe)}")
print(f"Num theses (diplom): {len(dataframe_diplom)}")
print()
print(dataframe_all["Grad"].value_counts())

Num theses (overall): 2982
Num theses (w/o diplom): 2126
Num theses (diplom): 856

Grad
Bachelor    1667
Diplom       856
Master       459
Name: count, dtype: int64


In [5]:
### Notice: Think about padding tokens for batch sizes > 1

vocab = set()
vocab.add("<pad>")

# For a more realistic application, we have to deal with unknown tokens
# that were not present in the training corpus. However, for the sake
# of clarity, we add all possible tokens from our dataset.
# vocab.add("<unk>")

# Prepare vocabulary
for s in dataframe_all.tokenized:
    vocab.update(s)

vocab_size = len(vocab)

word2idx = {w: idx for (idx, w) in enumerate(sorted(vocab))}
idx2word = {idx: w for (idx, w) in enumerate(sorted(vocab))}

print(f"Vocabulary size: {vocab_size}")

Vocabulary size: 7978


In [6]:
### PyTorch dataset for our thesis classification task

class ThesisClassificationDataset(Dataset):
    def __init__(self, dataset, labels, word2idx):
        self.data, self.labels = [], []
        for tokens, label in zip(dataset, labels):
            # Create inputs; map tokens to ids
            self.data.append(torch.stack([
                torch.tensor(word2idx[w], dtype=torch.long) for w in tokens
            ]))

            # Create labels; already an integer
            self.labels.append(label)


    def __len__(self):
        return len(self.data)


    def __getitem__(self, idx):
        # Returns one input and label sample
        return self.data[idx], self.labels[idx]

### Train and Evaluate

2.1 Implement the RNN for Classification. Therefore, you can use the nn.Module and overwrite the forward function.

2.2 Train and evaluate your models with 5-fold cross-validation. As in RNN-LM, you can either learn the embeddings from scratch or reuse the ones from word2vec.

2.3 Assemble a table: Recall/Precision/F1 measure for each of the mentioned RNN variants (RNN, GRU, LSTM). Which one works best?

2.4 Bonus: Apply your best classifier to the remaining diploma theses; are those on average more bachelor or master? :-)

In [7]:
### TODO: 2.1 Implement RNN classifier (nn.Module)
### Notice: Think about padding for batch sizes > 1
### Notice: 'torch.nn.utils.rnn' provides functionality

### YOUR CODE HERE

from torch.nn.utils.rnn import pack_padded_sequence, pad_sequence

class RNN_Classifier(nn.Module):
    rnn_dict = {
        "vanilla": nn.RNN, "gru": nn.GRU, "lstm": nn.LSTM
    }

    def __init__(self, num_embeddings, embedding_dim, hidden_dim, num_classes, 
                 rnn_variant, bidirectional=False, num_rnn_layers=1):
        super(RNN_Classifier, self).__init__()
        self.bidirectional = bidirectional
        self.num_directions = 2 if bidirectional else 1
        self.num_rnn_layers = num_rnn_layers

        self.embedding = nn.Embedding(
            num_embeddings=num_embeddings,
            embedding_dim=embedding_dim
        )

        self.rnn = self.rnn_dict[rnn_variant](
            input_size=embedding_dim,
            hidden_size=hidden_dim,
            num_layers=num_rnn_layers,
            bidirectional=bidirectional,
            dropout=0.2 if num_rnn_layers > 1 else 0,
        )

        # For bidrectional we have two times the hidden size;
        # forward + backward hidden states
        if bidirectional:
            print("Using bidirectional RNN!")
            hidden_dim *= 2

        self.fc = nn.Linear(hidden_dim, num_classes)

    
    def forward(self, X, lengths, hidden=None):
        embeddings = self.embedding(X)

        # Packed squence helps avoid unneccsary computation
        packed_seq = pack_padded_sequence(embeddings, lengths)

        outputs, hidden_states = self.rnn(packed_seq, hidden)

        # If tuple (h_n, c_n) containts cell state c_n then select h_n
        if isinstance(hidden_states, tuple):
            hidden = hidden_states[0]
        else:
            hidden = hidden_states

        # Get top hidden states of RNN stack (e.g., multiple layers)
        hidden_top = hidden.view(
            self.num_rnn_layers, 
            self.num_directions, 
            X.size(1), -1
        )[-1]

        # Consider bidirectional hidden states
        if self.bidirectional:
            h_forward_backward = torch.cat([hidden_top[0], hidden_top[1]], dim=-1)
            clf_input = h_forward_backward
        else:
            clf_input = hidden_top[-1]

        # Apply classifier with hidden states
        logits = self.fc(clf_input)

        return logits, hidden_states
    

class SequencePadder():
    def __init__(self, symbol) -> None:
        self.symbol = symbol


    def __call__(self, batch):
        # Processes the batch of samples into our desired format
        sorted_batch = sorted(batch, key=lambda x: x[0].size(0), reverse=True)
        sequences = [x[0] for x in sorted_batch]
        labels = [x[1] for x in sorted_batch]

        # Handles sequences with different lengths; pad the shorter ones
        # to match the length of the longest sequence in the batch
        padded = pad_sequence(sequences, padding_value=self.symbol)
        lengths = torch.LongTensor([len(x) for x in sequences])
        return padded, torch.LongTensor(labels), lengths


### END YOUR CODE

In [8]:
### TODO: 2.2 Implement the train functionality

### YOUR CODE HERE

def train(model, dataloader, criterion, optimizer, device):
    model.train()

    running_loss = 0.0

    optimizer.zero_grad()

    predictions = []
    ground_truth = []
    for inputs, labels, lengths in tqdm.tqdm(dataloader, desc="Train"):
        inputs = inputs.to(device)
        labels = labels.to(device)

        logits, hidden = model(inputs, lengths)

        preds = torch.argmax(logits, axis=-1)

        loss = criterion(logits, labels)

        loss.backward()
        optimizer.step()
        optimizer.zero_grad()

        running_loss += loss.item()
        predictions.extend(preds.cpu().numpy().tolist())
        ground_truth.extend(labels.cpu().numpy().tolist())

    running_loss = running_loss / len(dataloader)
    return predictions, ground_truth, running_loss

### END YOUR CODE

In [9]:
### TODO: 2.2 Implement the evaluation functionality

### YOUR CODE HERE

def eval(model, dataloader, criterion, device):
    model.eval()

    running_loss = 0.0

    predictions = []
    ground_truth = []
    with torch.no_grad():
        for inputs, labels, lengths in dataloader:
            inputs = inputs.to(device)
            labels = labels.to(device)

            logits, hidden = model(inputs, lengths)

            preds = torch.argmax(logits, axis=-1)

            loss = criterion(logits, labels)

            running_loss += loss.item()
            predictions.extend(preds.cpu().numpy().tolist())
            ground_truth.extend(labels.cpu().numpy().tolist())

    running_loss = running_loss / len(dataloader)
    return predictions, ground_truth, running_loss


def compute_metrics(preds, labels):
    return {
        'f1': sklearn_metrics.f1_score(y_true=labels, y_pred=preds),
        'prec': sklearn_metrics.precision_score(y_true=labels, y_pred=preds),
        'recall': sklearn_metrics.recall_score(y_true=labels, y_pred=preds),
        'acc': sklearn_metrics.accuracy_score(y_true=labels, y_pred=preds)
    }


### END YOUR CODE

In [10]:
### TODO: 2.3 Initialize and train the RNN Classification Model for X epochs + Evaluation

# Training parameters
SEED = 42
EPOCHS = 10
BATCH_SIZE = 16

LEARNING_RATE = 0.0001

DEVICE = "cpu" # 'cpu', 'mps' or 'cuda'
LABEL_COL = "label"
PAD_IDX = word2idx["<pad>"]

# Model parameters
EMBEDDING_DIM = 256
HIDDEN_DIM = 256
NUM_LAYERS = 1

### YOUR CODE HERE


folds = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED).split(
    dataframe.tokenized, dataframe[LABEL_COL].values
)


VARIANTS = [
    ("vanilla", False), ("gru", False), ("lstm", False),
    ("lstm", True) # Bidirectional flag
]


# Iterate variatants
results_all = []
models_all = {}
for rnn_variant, bidirectional in VARIANTS:
    model_name = rnn_variant
    if bidirectional:
        model_name += "-bidirectional"

    # Iterate folds
    for fold, (train_idx, test_idx) in enumerate(folds, start=1):
        train_data = dataframe.iloc[train_idx]
        test_data = dataframe.iloc[test_idx]

        # Oversampling for minority class
        train_data_master = train_data[train_data["Grad"] == "Master"]
        train_data = pd.concat([train_data, pd.DataFrame(train_data_master.to_dict('records')  * 2)])

        # Prepare samples
        train_labels = train_data[LABEL_COL].values
        test_labels = test_data[LABEL_COL].values
        train_data = train_data.tokenized
        test_data = test_data.tokenized

        # Use higher batch_size for training
        train_dataset = ThesisClassificationDataset(train_data, train_labels, word2idx=word2idx)
        train_dataloader = DataLoader(train_dataset, batch_size=BATCH_SIZE, collate_fn=SequencePadder(PAD_IDX))

        # Use batch_size=1 to avoid padding influence during testing
        test_dataset = ThesisClassificationDataset(test_data, test_labels, word2idx=word2idx)
        test_dataloader = DataLoader(test_dataset, batch_size=1, collate_fn=SequencePadder(PAD_IDX))

        model = RNN_Classifier(
            num_embeddings=len(vocab),
            embedding_dim=EMBEDDING_DIM,
            hidden_dim=HIDDEN_DIM,
            num_rnn_layers=NUM_LAYERS,
            rnn_variant=rnn_variant,
            bidirectional=bidirectional,
            num_classes=2
        )
        model = model.to(DEVICE)

        criterion = nn.CrossEntropyLoss(reduction="mean")

        optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)

        best_epoch = -1
        best_f1 = 0
        for epoch in range(1, EPOCHS + 1):
            print(f"Epoch {epoch} of {EPOCHS}")
            print("-" * 20)

            # Training step
            train_preds, train_labels, train_loss = train(
                model=model,
                dataloader=train_dataloader,
                criterion=criterion,
                optimizer=optimizer,
                device=DEVICE
            )

            # Evaluation step
            test_preds, test_labels, test_loss = eval(
                model=model,
                dataloader=test_dataloader,
                criterion=criterion,
                device=DEVICE
            )

            # Compute all metrics
            train_metrics = compute_metrics(preds=train_preds, labels=train_labels)
            test_metrics = compute_metrics(preds=test_preds, labels=test_labels)

            print(f"Train loss: {train_loss:.4f} | Train F1: {train_metrics['f1']:.4f}")
            print(f"Test loss: {test_loss:.4f} | Test F1: {test_metrics['f1']:.4f}")

            # Save best model by evaluation loss
            if best_f1 <= test_metrics["f1"]:
                best_epoch = epoch
                best_f1 = test_metrics["f1"]
                print(f"Saving best model ...")
                torch.save(model.state_dict(), f"data/best_{model_name}_clf.pt")

        # Load best and final model
        print(f"Best epoch: {best_epoch}")
        print(f"Best F1: {best_f1:.4f}")
        print(f"Loading best model ...")
        model.load_state_dict(torch.load(f"data/best_{model_name}_clf.pt"))

        # Compute test metrics
        test_preds, test_labels, test_loss = eval(
            model=model, dataloader=test_dataloader, criterion=criterion, device=DEVICE
        )
        test_metrics = compute_metrics(preds=test_preds, labels=test_labels)
        results_all.append({"variant": model_name, **test_metrics})
        print(test_metrics)

        models_all[model_name] = model

        # Remove for cross-validation results
        break

### END YOUR CODE

Epoch 1 of 10
--------------------


Train: 100%|██████████| 153/153 [00:01<00:00, 104.43it/s]


Train loss: 0.6882 | Train F1: 0.4600
Test loss: 0.7801 | Test F1: 0.2909
Saving best model ...
Epoch 2 of 10
--------------------


Train: 100%|██████████| 153/153 [00:01<00:00, 104.57it/s]


Train loss: 0.6308 | Train F1: 0.6365
Test loss: 0.7819 | Test F1: 0.2979
Saving best model ...
Epoch 3 of 10
--------------------


...

Train loss: 0.1816 | Train F1: 0.9516
Test loss: 0.9613 | Test F1: 0.3188
Epoch 9 of 10
--------------------


Train: 100%|██████████| 153/153 [00:03<00:00, 38.29it/s]


Train loss: 0.1054 | Train F1: 0.9776
Test loss: 1.0077 | Test F1: 0.3125
Epoch 10 of 10
--------------------


Train: 100%|██████████| 153/153 [00:04<00:00, 37.24it/s]


Train loss: 0.0552 | Train F1: 0.9950
Test loss: 1.0732 | Test F1: 0.2845
Best epoch: 2
Best F1: 0.3395
Loading best model ...
{'f1': 0.3395061728395062, 'prec': 0.23706896551724138, 'recall': 0.5978260869565217, 'acc': 0.4964705882352941}


In [11]:
# Assemble table with results
results_df = pd.DataFrame(results_all)
results_df = results_df.groupby("variant").mean()
results_df

,f1,prec,recall,acc
variant,,,,
gru,0.325301,0.224066,0.593407,0.472941
lstm,0.305344,0.235294,0.434783,0.571765
lstm-bidirectional,0.339506,0.237069,0.597826,0.496471
vanilla,0.304207,0.216590,0.510870,0.495305


In [12]:
### TODO: 2.4 Apply your best classifier to the remaining diploma theses

# Best model selection
best_model_name = results_df["f1"].idxmax()
best_model = models_all[best_model_name]
print(f"Best classifier: {best_model_name}")

# Prepare diploma test dataset
diplom_tokenized, diplom_labels = dataframe_diplom.tokenized, dataframe_diplom[LABEL_COL].values
diplom_labels = np.ones_like(diplom_labels)
diplom_dataset = ThesisClassificationDataset(diplom_tokenized, diplom_labels, word2idx=word2idx)
diplom_dataloader = DataLoader(diplom_dataset, batch_size=1, collate_fn=SequencePadder(PAD_IDX))

# Prediction step
diplom_preds, _, _ = eval(
    model=best_model, dataloader=diplom_dataloader, criterion=criterion, device=DEVICE
)

bachelor_preds = diplom_preds.count(LABEL2IDX["Bachelor"])
master_preds = diplom_preds.count(LABEL2IDX["Master"])

# Check thesis match
print("Predictions for diploma thesis titles:")
print(f"Diplom -> Bachelor ({round((bachelor_preds / len(diplom_preds)) * 100, 2)} %)")
print(f"Diplom -> Master ({round((master_preds / len(diplom_preds)) * 100, 2)} %)")

Best classifier: lstm-bidirectional
Predictions for diploma thesis titles:
Diplom -> Bachelor (46.03 %)
Diplom -> Master (53.97 %)
